# Molmo 2: Pointing on Satellite Imagery (via Ollama)

**Goal:** Test whether Molmo 2 can **point to** hydraulic features on satellite imagery with pixel coordinates.

**Context:** In attempt1, Molmo 2 was blocked — unavailable on OpenRouter. Now we run it **locally via Ollama**, which serves quantized weights that fit on M1 Pro 32GB.

Molmo 2 is the only VLM that returns native `(x, y)` point coordinates, making it the ideal bridge to SAM segmentation.

| Model | Capability | Output |
|-------|-----------|--------|
| Gemma 4 | Describe features | Qualitative text |
| **Molmo 2** | **Point to features** | **`<point x="52.3" y="41.7">river channel</point>`** |
| SAM (Stage 7) | Segment from points | Binary mask |

**Roadmap reference:** Stage 6.4 — Molmo 2 Pointing on Satellite (Local)

## Tasks
1. Connect to Molmo 2 via Ollama (local)
2. Test pointing prompts on satellite RGB
3. Parse point coordinates and overlay on satellite image
4. Convert pixel coordinates to geo-coordinates
5. Evaluate: are points accurate enough for SAM prompts?
6. Save timestamped outputs to `data/output/model-outputs/attempt2/molmo2/`

In [ ]:
import os
import base64
import io
import re
import json
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from PIL import Image
from openai import OpenAI

# Paths
DEM_PATH = Path("../../data/input/1m elevation.tif")
SATELLITE_PATH = Path("../../data/output/cimarron_satellite.png")
HILLSHADE_PATH = Path("../../data/output/cimarron_hillshade.png")
OUTPUT_DIR = Path("../../data/output/model-outputs/attempt2/molmo2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Timestamp for this run
RUN_TS = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

print(f"DEM: {DEM_PATH}")
print(f"Satellite image: {SATELLITE_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Run timestamp: {RUN_TS}")

## 1.1 Connect to Molmo 2 via Ollama

Ollama exposes an OpenAI-compatible API at `http://localhost:11434/v1`.

**Setup:** Run `ollama pull molmo2` in your terminal before starting this notebook.

In [ ]:
# Ollama serves an OpenAI-compatible API locally
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Ollama doesn't need a real key
)

MODEL = "molmo2"
MOLMO_AVAILABLE = False

try:
    test = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Say hello in one word."}],
        max_tokens=10,
    )
    print(f"Molmo 2 (Ollama) connection OK: {test.choices[0].message.content}")
    print(f"Model: {MODEL}")
    MOLMO_AVAILABLE = True
except Exception as e:
    print(f"Molmo 2 is NOT available via Ollama.")
    print(f"Error: {e}")
    print()
    print("Setup instructions:")
    print("  1. Install Ollama: https://ollama.com")
    print("  2. Pull the model: ollama pull molmo2")
    print("  3. Verify it's running: ollama list")
    print()
    print("The rest of this notebook will be skipped.")

## 1.2 Load Images & DEM Metadata

In [ ]:
# Load DEM metadata for geo-coordinate conversion
with rasterio.open(DEM_PATH) as src:
    dem_transform = src.transform
    dem_bounds = src.bounds
    dem_crs = src.crs

print(f"CRS: {dem_crs}")
print(f"Bounds: {dem_bounds}")

# Load satellite image
if not SATELLITE_PATH.exists():
    raise FileNotFoundError(
        f"Satellite image not found at {SATELLITE_PATH}.\n"
        "Run the satellite acquisition notebook (Stage 5) first."
    )

img_satellite = Image.open(SATELLITE_PATH)
print(f"Satellite image: {img_satellite.size} ({img_satellite.mode})")

# Load hillshade for dual-input comparison
img_hillshade = None
if HILLSHADE_PATH.exists():
    img_hillshade = Image.open(HILLSHADE_PATH)
    print(f"Hillshade image: {img_hillshade.size} ({img_hillshade.mode})")

## 1.3 Inference & Coordinate Parsing Utilities

In [ ]:
def resize_for_api(img, max_width=1024):
    """Resize image to reduce memory while preserving aspect ratio."""
    if img.width > max_width:
        ratio = max_width / img.width
        new_size = (max_width, int(img.height * ratio))
        return img.resize(new_size, Image.LANCZOS)
    return img


def image_to_base64(img):
    """Convert PIL Image to base64 data URL."""
    img = resize_for_api(img)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{b64}"


def vision_query(image, prompt, max_tokens=512):
    """Send an image + text prompt to Molmo 2 via Ollama."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_to_base64(image)}},
                ],
            }
        ],
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


def parse_molmo_points(text, image_w, image_h):
    """Parse Molmo coordinate output into pixel coordinates.

    Handles multiple formats:
    - Legacy (0-100 scale): <point x=\"10.0\" y=\"20.0\">label</point>
    - Molmo 2 (x1000 scale): <points coords=\"x1,y1,x2,y2,...\"/>

    Returns list of (x_px, y_px, label) tuples.
    """
    points = []

    # Legacy format: <point x=\"...\" y=\"...\">label</point>
    legacy_pattern = r'<point\s+x="([\d.]+)"\s+y="([\d.]+)"(?:\s+alt="[^"]*")?>([^<]*)</point>'
    for match in re.finditer(legacy_pattern, text):
        x_pct, y_pct, label = float(match.group(1)), float(match.group(2)), match.group(3).strip()
        x_px = x_pct / 100.0 * image_w
        y_px = y_pct / 100.0 * image_h
        points.append((x_px, y_px, label))

    # Molmo 2 format: <points coords=\"x1,y1,x2,y2,...\"/>
    coords_pattern = r'<points\s+coords="([^"]+)"\s*/>'
    for match in re.finditer(coords_pattern, text):
        coords = [float(c) for c in match.group(1).split(",")]
        for i in range(0, len(coords) - 1, 2):
            x_px = coords[i] / 1000.0 * image_w
            y_px = coords[i + 1] / 1000.0 * image_h
            points.append((x_px, y_px, f"point_{i // 2}"))

    return points


def pixels_to_geo(points, transform):
    """Convert pixel coordinates to geo-coordinates."""
    geo_coords = []
    for x_px, y_px, label in points:
        easting, northing = rasterio.transform.xy(transform, int(y_px), int(x_px))
        geo_coords.append((easting, northing, label))
    return geo_coords


def molmo_point(image, prompt):
    """Run a pointing query and parse the results."""
    resized = resize_for_api(image)
    raw = vision_query(image, prompt)
    pixel_coords = parse_molmo_points(raw, resized.width, resized.height)
    geo_coords = pixels_to_geo(pixel_coords, dem_transform) if pixel_coords else []
    return {
        "raw_output": raw,
        "pixel_coords": pixel_coords,
        "geo_coords": geo_coords,
        "n_points": len(pixel_coords),
    }


def plot_points_on_image(img, points_px, title, save_path=None, figsize=(14, 8)):
    """Overlay parsed points on an image."""
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(img)
    ax.set_title(title, fontsize=14)

    if points_px:
        xs = [p[0] for p in points_px]
        ys = [p[1] for p in points_px]
        labels = [p[2] for p in points_px]
        ax.scatter(xs, ys, c="red", s=80, marker="x", linewidths=2, zorder=5)
        for x, y, label in zip(xs, ys, labels):
            ax.annotate(label, (x, y), textcoords="offset points",
                        xytext=(5, 5), fontsize=8, color="red",
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))
        ax.set_xlabel(f"{len(points_px)} points detected")
    else:
        ax.set_xlabel("No coordinate points parsed from model output")

    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()
    return fig


pointing_results = {}
print("Utilities ready.")

## 2.1 Pointing Tests — Satellite

In [ ]:
if not MOLMO_AVAILABLE:
    print("Skipping — Molmo 2 is not available.")
else:
    satellite_pointing_prompts = {
        "channel": "Point to the main river channel in this satellite image.",
        "channel_bends": "Point to where the river channel bends or meanders in this satellite image.",
        "roads": "Point to any roads or road embankments visible in this satellite image.",
        "vegetation_boundary": "Point to the boundaries between forest and open grassland in this satellite image.",
        "point_bars": "Point to any sand bars or point bars along the river in this satellite image.",
        "bridges": "Point to any bridges crossing the river in this satellite image.",
    }

    for key, prompt in satellite_pointing_prompts.items():
        test_name = f"satellite_{key}"
        print(f"\n{'='*80}")
        print(f"TEST: {test_name}")
        print(f"Prompt: {prompt}")
        print(f"{'='*80}")

        result = molmo_point(img_satellite, prompt)
        pointing_results[test_name] = {"prompt": prompt, **result}

        print(f"\nRaw output:\n{result['raw_output']}")
        print(f"\nPoints found: {result['n_points']}")
        if result['pixel_coords']:
            for px, py, label in result['pixel_coords']:
                print(f"  ({px:.0f}, {py:.0f}) — {label}")

        save_path = OUTPUT_DIR / f"{RUN_TS}_satellite_{key}.png"
        plot_points_on_image(img_satellite, result['pixel_coords'],
                             f"Molmo 2 Pointing: {key} (satellite)",
                             save_path=save_path)

## 3. Results Summary & Save

In [ ]:
if not MOLMO_AVAILABLE:
    print("Molmo 2 was not available — no results to summarize.")
else:
    # Pointing results table
    print("POINTING RESULTS")
    print(f"{'Test':<30} {'Points':>6}  {'Coordinates (pixel)'}")
    print("-" * 80)
    for key, result in pointing_results.items():
        n = result["n_points"]
        if result["pixel_coords"]:
            coords_str = "; ".join(f"({p[0]:.0f},{p[1]:.0f})" for p in result["pixel_coords"][:5])
            if n > 5:
                coords_str += f" ... (+{n - 5} more)"
        else:
            coords_str = "(no coordinates parsed)"
        print(f"{key:<30} {n:>6}  {coords_str}")

    print(f"\nModel: {MODEL} (via Ollama, local)")
    print(f"Total pointing tests: {len(pointing_results)}")

    # Save results to markdown
    md_lines = [
        f"# Molmo 2 — Satellite Pointing Test Results\n",
        f"\n**Date:** {RUN_TS}\n",
        f"\n**Model:** `{MODEL}` (via Ollama, local)\n",
        f"\n**Input:** Satellite (NAIP RGB, 0.6m/pixel)\n",
        f"\n**DEM:** Cimarron River, 1m resolution, {dem_crs}\n",
        f"\n**Notebook:** `notebooks/attempt2/02_molmo2_satellite.ipynb`\n",
        f"\n**Attempt:** 2 (satellite imagery, Ollama local inference)\n",
        "\n---\n",
    ]

    for key, result in pointing_results.items():
        md_lines.append(f"\n## Pointing: {key.replace('_', ' ').title()}\n")
        md_lines.append(f"\n**Prompt:** {result['prompt']}\n")
        md_lines.append(f"\n**Points found:** {result['n_points']}\n")
        if result["pixel_coords"]:
            md_lines.append("\n**Pixel coordinates:**\n\n")
            for px, py, label in result["pixel_coords"]:
                md_lines.append(f"- ({px:.0f}, {py:.0f}) — {label}\n")
        if result["geo_coords"]:
            md_lines.append(f"\n**Geo-coordinates ({dem_crs}):**\n\n")
            for e, n, label in result["geo_coords"]:
                md_lines.append(f"- ({e:.1f}, {n:.1f}) — {label}\n")
        md_lines.append(f"\n**Raw output:**\n\n```\n{result['raw_output']}\n```\n")
        md_lines.append("\n---\n")

    # Summary table
    md_lines.append("\n## Summary\n\n")
    md_lines.append("| Test | Points | Notes |\n")
    md_lines.append("|------|--------|-------|\n")
    for key, result in pointing_results.items():
        n = result["n_points"]
        note = "coordinates parsed" if n > 0 else "no coordinates in output"
        md_lines.append(f"| {key} | {n} | {note} |\n")

    output_path = OUTPUT_DIR / f"{RUN_TS}_molmo2_satellite_results.md"
    with open(output_path, "w") as f:
        f.writelines(md_lines)

    print(f"\nResults saved to: {output_path}")